# OCR Bilans Fiscaux Algériens — V10 — JSON structuré complet

> Objectif : 1 PDF → 1 JSON avec ACTIF, PASSIF, TCR, DECL et TOUTES les annexes en schémas canoniques structurés (mêmes principes que ACTIF/PASSIF/TCR).

## Journal des versions
| Version | Date | Changement |
|---|---|---|
| V10 | 2026-08-16 | Annexes 1→13 + 8/1 + 8/2 + sous-traitance + rémunérations + répartition en schémas canoniques ; classification 100% contenu (titres numérotés N/, jamais l'ordre) ; valeurs brutes (pas de logique incertain dans cette phase) ; progression live ; pages blanches sans VLM ; generate(**inputs) ; MAX_NEW_TOKENS 4096 ; GPU_BATCH_SIZE 4 |

## Règle d'or
Ne jamais inventer de valeur : absent/illisible → null.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 1 — DEPENDANCES | BILANS_V10
# Objectif: installer le runtime minimal pour produire les JSON.
# Entrees: environnement Python.
# Sorties: librairies installees.
# Regles: pas openpyxl, pas pandas, pas export Excel.
# ════════════════════════════════════════════════════════════
%pip install -q -U 'transformers>=4.57.0' accelerate pymupdf pillow psutil
print('✅ Dependances OK')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 2 — IMPORTS | BILANS_V10
# Objectif: charger les librairies du pipeline JSON.
# Entrees: librairies installees.
# Sorties: namespaces importes.
# Regles: aucun import Excel/controles.
# ════════════════════════════════════════════════════════════
import time, json, re, gc, copy, unicodedata
import numpy as np
from pathlib import Path
from datetime import datetime
import fitz, torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
print('✅ Imports OK')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 3 — CONFIG | BILANS_V10
# Objectif: chemins, parametres modele, batch, sorties JSON.
# Entrees: dossier PDF.
# Sorties: liste pdfs, chemins JSON_DIR/LOG/INDEX.
# Regles: JSON uniquement; generation plafonnee; batch 4.
# ════════════════════════════════════════════════════════════
MODEL_PATH = '/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.6-27B-FP8/main'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MAX_NEW_TOKENS = 4096
IMAGE_MAX_SIZE = 2024
MIN_PIXELS = 4 * 32 * 32
MAX_PIXELS = 2000 * 32 * 32
PDF_ZOOM = 3.0
BLANK_THRESHOLD = 0.95
CLASSIF_BATCH_SIZE = 16
GPU_BATCH_SIZE = 4
INPUT_DIR = Path('/mnt/Risk/bilans_in')
OUTPUT_DIR = Path('/mnt/Risk/bilans_out')
JSON_DIR = OUTPUT_DIR / 'json_bilans_v10'
LOG_PATH = OUTPUT_DIR / 'pipeline_bilans_v10.log'
INDEX_PATH = OUTPUT_DIR / 'index_bilans_v10.jsonl'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
JSON_DIR.mkdir(parents=True, exist_ok=True)
pdfs = sorted(INPUT_DIR.glob('*.pdf'))
print('Device: ' + DEVICE + ' | PDF: ' + str(len(pdfs)) + ' | JSON: ' + str(JSON_DIR))

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 4 — LOG | BILANS_V10
# Objectif: journaliser avec affichage immediat (flush) pour suivi live.
# Entrees: message.
# Sorties: ligne horodatee stdout + fichier.
# ════════════════════════════════════════════════════════════
def log(msg):
    ligne = datetime.now().strftime('%Y-%m-%d %H:%M:%S') + ' — ' + str(msg)
    print(ligne, flush=True)
    with open(LOG_PATH, 'a', encoding='utf-8') as f:
        f.write(ligne + chr(10))
print('✅ Log OK')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 5 — CHARGEMENT MODELE | BILANS_V10
# Objectif: charger Qwen3.6-VL FP8 dequantise en bf16.
# Entrees: MODEL_PATH.
# Sorties: processor, model.
# Regles: padding gauche; generation deterministe; thinking off.
# ════════════════════════════════════════════════════════════
t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
processor.tokenizer.padding_side = 'left'
try:
    from transformers.integrations.finegrained_fp8 import FineGrainedFP8Config as FP8Config
except ImportError:
    from transformers import FineGrainedFP8Config as FP8Config
model = AutoModelForImageTextToText.from_pretrained(MODEL_PATH, dtype=torch.bfloat16, device_map='auto', trust_remote_code=True, low_cpu_mem_usage=True, quantization_config=FP8Config(dequantize=True))
model.eval()
print('✅ Modele charge en ' + str(round(time.time()-t0, 1)) + 's')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 6 — UTILITAIRES | BILANS_V10
# Objectif: rendu PDF, deskew, pages blanches, inference VLM batch.
# Entrees: PDF, images.
# Sorties: pages images, reponses VLM.
# Regles: pages blanches detectees localement, non envoyees au VLM;
#         generate(**inputs) obligatoire (regression v8bis corrigee).
# ════════════════════════════════════════════════════════════
def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def resize(img, max_side=IMAGE_MAX_SIZE):
    w, h = img.size
    if max(w, h) <= max_side:
        return img
    r = max_side / max(w, h)
    return img.resize((int(w*r), int(h*r)), Image.LANCZOS)

def strip_accents(s):
    s = str(s)
    return ''.join(c for c in unicodedata.normalize('NFKD', s) if not unicodedata.combining(c))

def norm_key(s):
    s = strip_accents(str(s)).lower()
    s = re.sub('[^a-z0-9]+', '_', s)
    return s.strip('_')

def estimate_skew(img):
    small = img.convert('L').copy()
    small.thumbnail((500, 500))
    def score(a):
        r = np.array(small.rotate(a, expand=True, fillcolor=255)) < 128
        proj = r.sum(axis=1)
        return float((proj ** 2).sum())
    best = max(range(-12, 13, 2), key=score)
    best = max([best-1, best-0.5, best, best+0.5, best+1], key=score)
    return best if abs(best) >= 1 else 0.0

def deskew(img):
    a = estimate_skew(img)
    if a:
        img = img.rotate(a, expand=True, fillcolor=(255,255,255), resample=Image.BICUBIC)
    return img, a

def is_blank(image, threshold=BLANK_THRESHOLD):
    arr = np.array(image.convert('L'))
    return (arr > 240).sum() / arr.size >= threshold

def pdf_to_pages(path, zoom=PDF_ZOOM):
    doc = fitz.open(path)
    matrix = fitz.Matrix(zoom, zoom)
    pages = []
    for i in range(len(doc)):
        pix = doc.load_page(i).get_pixmap(matrix=matrix, alpha=False)
        img = Image.frombytes('RGB', [pix.width, pix.height], pix.samples)
        img, angle = deskew(img)
        img = resize(img)
        pages.append({'index': i, 'image': img, 'largeur_px': img.width, 'hauteur_px': img.height, 'rotation_estimee': angle})
    doc.close()
    return pages

def parse_json(text):
    if not text:
        return {}
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    start = text.find('{')
    end = text.rfind('}')
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end+1])
        except Exception:
            return {}
    return {}

def apply_template(messages):
    try:
        return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def _decode(out_i, in_len):
    return processor.decode(out_i[in_len:], skip_special_tokens=True, clean_up_tokenization_spaces=False)

def ask_single(prompt, image):
    msgs = [{'role':'user','content':[{'type':'image','image':image},{'type':'text','text':prompt}]}]
    inputs = processor(text=[apply_template(msgs)], images=[image], return_tensors='pt').to(DEVICE)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, repetition_penalty=1.0, pad_token_id=processor.tokenizer.eos_token_id)
    torch.cuda.synchronize()
    return {'text': _decode(out[0], inputs['input_ids'].shape[1]), 'tokens_in': int(inputs['input_ids'].shape[1]), 'tokens_out': int(out[0].shape[0] - inputs['input_ids'].shape[1]), 'elapsed': round(time.time()-t0, 2)}

def ask_batch(prompt, images):
    if not images:
        return []
    if len(images) == 1:
        return [ask_single(prompt, images[0])]
    msgs = [[{'role':'user','content':[{'type':'image','image':img},{'type':'text','text':prompt}]}] for img in images]
    inputs = processor(text=[apply_template(m) for m in msgs], images=images, return_tensors='pt', padding=True).to(DEVICE)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, repetition_penalty=1.0, pad_token_id=processor.tokenizer.eos_token_id)
    torch.cuda.synchronize()
    el = time.time() - t0
    in_len = inputs['input_ids'].shape[1]
    attn = inputs.get('attention_mask')
    return [{'text': _decode(out[i], in_len), 'tokens_in': int(attn[i].sum().item()) if attn is not None else in_len, 'tokens_out': int(out[i].shape[0] - in_len), 'elapsed': round(el/len(images), 2)} for i in range(len(images))]

print('✅ Utilitaires OK')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 7 — SCHEMAS CANONIQUES + CLASSIFICATION CONTENU + PROMPTS | BILANS_V10
# Objectif: contrat de donnees unique pour ACTIF/PASSIF/TCR/DECL
#           ET toutes les annexes (1→13, 8/1, 8/2, ST, REM, DIST).
# Regles: classification par contenu imprime (titres N/), jamais par ordre.
# ════════════════════════════════════════════════════════════
SCHEMAS = {
 'ACTIF': {'titre': 'BILAN (ACTIF)', 'cols': ['montant_brut', 'amortissements_provisions_pertes', 'net_n', 'net_n1'], 'postes': {
   'ecarts_acquisition_goodwill': 'Ecart d acquisition goodwill',
   'immobilisations_incorporelles': 'Immobilisations incorporelles',
   'terrains': 'Terrains',
   'batiments': 'Batiments',
   'autres_immobilisations_corporelles': 'Autres Immobilisations corporelles',
   'immobilisations_en_concession': 'Immobilisations en concession',
   'immobilisations_en_cours': 'Immobilisations en cours',
   'titres_mis_en_equivalence': 'Titres mis en equivalence',
   'autres_participations_creances': 'Autres participations et creances rattachees',
   'autres_titres_immobilises': 'Autres titres immobilises',
   'prets_actifs_financiers_non_courants': 'Prets et autres actifs financiers non courants',
   'impots_differes_actif': 'Impots Differes Actif',
   'total_actif_non_courant': 'TOTAL ACTIF NON COURANT',
   'stocks_encours': 'Stocks et encours',
   'clients': 'Clients',
   'autres_debiteurs': 'Autres debiteurs',
   'impots_assimiles_actif': 'Impots et assimiles',
   'autres_creances_assimiles': 'Autres Creances et Emplois assimiles',
   'placements_financiers_courants': 'Placements et autres actifs financiers courants',
   'tresorerie_actif': 'Tresorerie',
   'total_actif_courant': 'TOTAL ACTIF COURANT',
   'total_general_actif': 'TOTAL GENERAL ACTIF'}},
 'PASSIF': {'titre': 'BILAN (PASSIF)', 'cols': ['n', 'n1'], 'postes': {
   'capital_emis': 'Capital emis',
   'capital_non_appele': 'Capital non appele',
   'primes_reserves': 'Primes et reserves',
   'ecart_reevaluation': 'Ecart de reevaluation',
   'ecart_equivalence': 'Ecart d equivalence',
   'resultat_net_passif': 'Resultat net',
   'report_a_nouveau': 'Report a nouveau',
   'part_societe_consolidante': 'Part de la societe consolidante',
   'part_minoritaires': 'Part des minoritaires',
   'total_capitaux_propres': 'TOTAL I',
   'emprunts_dettes_financieres': 'Emprunts et dettes financieres',
   'impots_differes_provisionnes': 'Impots differes et provisionnes',
   'autres_dettes_non_courantes': 'Autres dettes non courantes',
   'provisions_produits_avance': 'Provisions et produits constatés d avance',
   'total_passifs_non_courants': 'TOTAL PASSIFS NON COURANTS II',
   'fournisseurs_rattaches': 'Fournisseurs et comptes rattaches',
   'impots_passif': 'Impots',
   'autres_dettes': 'Autres dettes',
   'tresorerie_passif': 'Tresorerie Passif',
   'total_passifs_courants': 'TOTAL PASSIFS COURANTS',
   'total_general_passif': 'TOTAL GENERAL PASSIF'}},
 'TCR': {'titre': 'COMPTE DE RESULTAT', 'cols': ['n_debit', 'n_credit', 'n1_debit', 'n1_credit'], 'postes': {
   'ventes_marchandises': 'Ventes de Marchandises',
   'produits_fabriques': 'Produits Fabriques',
   'prestations_services': 'Prestations de Services',
   'ventes_travaux': 'Ventes de Travaux',
   'produits_annexes': 'Produits Annexes',
   'rabais_remises_ristournes_accordes': 'Rabais remises ristournes accordes',
   'chiffre_affaires_net': 'Chiffre d affaires net',
   'production_stockee_destockee': 'Production Stockee ou destockee',
   'production_immobilisee': 'Production immobilisee',
   'subvention_exploitation': 'Subvention d exploitation',
   'production_exercice': 'I-Production de l exercice',
   'achats_marchandises_vendues': 'Achats de Marchandises vendues',
   'matieres_premieres': 'Matieres premieres',
   'autres_approvisionnements': 'Autres Approvisionnements',
   'variation_stocks': 'Variation des Stocks',
   'achats_etudes_prestations': 'Achats d Etudes et de Prestations de services',
   'autres_consommations': 'Autres consommations',
   'sous_traitance_generale': 'Sous-traitance generale',
   'locations': 'Locations',
   'entretien_reparations': 'Entretien reparations et maintenance',
   'primes_assurances': 'Primes d assurances',
   'personnel_exterieur': 'Personnel exterieur a l entreprise',
   'remuneration_intermediaires': 'Remuneration d intermediaires et honoraires',
   'publicite': 'Publicite',
   'deplacements_missions': 'Deplacements missions et receptions',
   'autres_services': 'Autres services',
   'consommations_exercice': 'II-Consommations de l exercice',
   'valeur_ajoutee_exploitation': 'III-Valeur ajoutee d exploitation',
   'charges_personnel': 'Charges de personnel',
   'impots_taxes_assimiles': 'Impots et taxes et versements assimiles',
   'excedent_brut_exploitation': 'IV-Excedent brut d exploitation',
   'autres_produits_operationnels': 'Autres produits operationnels',
   'autres_charges_operationnelles': 'Autres charges operationnelles',
   'dotations_amortissements': 'Dotations aux amortissements',
   'provisions': 'Provisions',
   'pertes_valeur': 'Perte de Valeur',
   'reprises_pertes_valeur_provisions': 'Reprise sur pertes de valeur et provisions',
   'resultat_operationnel': 'V-Resultat operationnel',
   'produits_financiers': 'Produits financiers',
   'charges_financieres': 'Charges financieres',
   'resultat_financier': 'VI-Resultat Financier',
   'resultat_ordinaire': 'VII-Resultat ordinaire',
   'elements_extraordinaires_produits': 'Elements extraordinaires Produits',
   'elements_extraordinaires_charges': 'Elements extraordinaires Charges',
   'resultat_extraordinaire': 'VIII-Resultat extraordinaire',
   'impots_exigibles_resultats': 'Impots exigibles sur resultats',
   'impots_differes_resultats': 'Impots differes sur resultats',
   'resultat_net_exercice': 'RESULTAT NET DE L EXERCICE'}},
 'DECL': {'titre': 'DECLARATION IBS', 'cols': ['valeur'], 'kinds': {'nif': 'nif', 'raison_sociale': 'texte', 'activite_principale': 'texte', 'registre_commerce': 'texte', 'adresse_siege': 'texte', 'cac_cabinet': 'texte', 'cac_nom': 'texte', 'exercice_annee': 'annee', 'annee_souscription': 'annee'}, 'postes': {
   'nif': 'Numero d Identification Fiscale',
   'raison_sociale': 'Designation de l entreprise',
   'activite_principale': 'Activite principale',
   'registre_commerce': 'Registre de Commerce',
   'adresse_siege': 'Adresse siege social',
   'cac_cabinet': 'Certification des comptes - Cabinet',
   'cac_nom': 'Certification des comptes - Nom CAC',
   'exercice_annee': 'Resultat de l exercice - Annee',
   'annee_souscription': 'Annee de souscription',
   'chiffre_affaires_global_ht': 'Chiffre d affaires global hors taxes',
   'resultat_comptable': 'Resultat comptable',
   'resultat_fiscal': 'Resultat fiscal'}},
 'A1': {'titre': '1/ Tableau des mouvements des stocks', 'cols': ['solde_debut', 'debit', 'credit', 'solde_fin'], 'postes': {
   'stocks_marchandises': 'Stocks de marchandises',
   'matieres_fournitures': 'Matieres et fournitures',
   'autres_approvisionnements': 'Autres approvisionnements',
   'encours_production_biens': 'Encours de production de biens',
   'encours_production_services': 'Encours de production de services',
   'stocks_produits': 'Stocks de produits',
   'stocks_provenant_immobilisations': 'Stocks provenant d immobilisations',
   'stocks_exterieur': 'Stocks a l exterieur',
   'total': 'TOTAL'}},
 'A2': {'titre': '2/ Tableau de la fluctuation de la production stockee', 'cols': ['debit', 'credit', 'solde_debiteur', 'solde_crediteur'], 'postes': {
   'production_stockee': 'Production stockee ou destockee (ligne unique si non libellee)'}},
 'A3': {'titre': '3/ Charges de personnel, impots, taxes, autres services', 'cols': ['montant'], 'postes': {
   'charges_locatives': 'Charges locatives et charges de copropriete',
   'etudes_recherches': 'Etudes et recherches',
   'documentation_divers': 'Documentation et divers',
   'transports_biens': 'Transports de biens et transport collectif du personnel',
   'frais_postaux': 'Frais postaux et de telecommunications',
   'services_bancaires': 'Services bancaires et assimiles',
   'cotisations_divers': 'Cotisations et divers',
   'total_autres_services': 'TOTAL (1)',
   'remunerations_personnel': 'Remunerations du personnel',
   'remuneration_exploitant': 'Remunerations de l exploitant individuel (cas d une EURL)',
   'cotisations_sociales': 'Cotisations aux organismes sociaux',
   'charges_sociales_exploitant': 'Charges sociales de l exploitant individuel (cas d une EURL)',
   'autres_charges_sociales': 'Autres charges sociales',
   'autres_charges_personnel': 'Autres charges de personnels',
   'total_charges_personnel': 'TOTAL (2)',
   'impots_sur_remunerations': 'Impots taxes et versements assimiles sur remunerations',
   'impots_non_recuperables': 'Impots et taxes non recuperables sur chiffres d affaires',
   'autres_impots_taxes': 'Autres impots et taxes (hors impots sur les resultats)',
   'total_impots': 'TOTAL (3)',
   'total_general': 'TOTAL (1)+(2)+(3)'}},
 'A4': {'titre': '4/ Autres charges et produits operationnels', 'cols': ['montant'], 'postes': {
   'redevances_concessions_charges': 'Redevances pour concessions brevets licences logiciels (charges)',
   'moins_values_sorties_actifs': 'Moins values sur sorties d actifs immobilises non financiers',
   'jetons_presence_charges': 'Jetons de presence (charges)',
   'pertes_creances_irrecouvrables': 'Perte sur creances irrecouvrables',
   'quote_part_operations_commun_charges': 'Quote-part de resultat sur operations faites en commun (charges)',
   'amendes_penalites_dons': 'Amendes et penalites subventions accordees dons et liberalites',
   'charges_exceptionnelles_gestion': 'Charges exceptionnelles de gestion courante',
   'autres_charges_gestion': 'Autres charges de gestion courante',
   'total_charges': 'TOTAL (charges)',
   'redevances_concessions_produits': 'Redevances pour concessions brevets licences logiciels (produits)',
   'plus_values_sorties_actifs': 'Plus values sur sorties d actifs immobilises non financiers',
   'jetons_presence_produits': 'Jetons de presence et remunerations d administrateurs ou de gerant',
   'quotes_parts_subventions_virees': 'Quotes-parts de subventions d investissement virees au resultat',
   'quote_part_operations_commun_produits': 'Quote-part de resultat sur operations faites en commun (produits)',
   'rentrees_creances_amorties': 'Rentree sur creances amorties',
   'produits_exceptionnels_gestion': 'Produits exceptionnels sur operations de gestion',
   'autres_produits_gestion': 'Autres produits de gestion courante',
   'total_produits': 'TOTAL (produits)'}},
 'A5': {'titre': '5/ Tableau des amortissements et pertes de valeurs', 'cols': ['dotations_cumulees_debut', 'dotations_exercice', 'diminutions_elements_sortis', 'dotations_cumulees_fin', 'dotations_fiscales_exercice', 'ecarts'], 'postes': {
   'goodwill': 'Goodwill',
   'immobilisations_incorporelles': 'Immobilisations incorporelles',
   'immobilisations_corporelles': 'Immobilisations corporelles',
   'participations': 'Participations',
   'autres_actifs_financiers_non_courants': 'Autres actifs financiers non courants',
   'total': 'TOTAL'}},
 'A6': {'titre': '6/ Tableau des immobilisations creees ou acquises', 'cols': ['montants_bruts', 'tva_deduite', 'montant_net_a_amortir'], 'postes': {
   'goodwill': 'Goodwill',
   'immobilisations_incorporelles': 'Immobilisations incorporelles',
   'immobilisations_corporelles': 'Immobilisations corporelles',
   'participations': 'Participations',
   'autres_actifs_financiers_non_courants': 'Autres actifs financiers non courants',
   'total': 'TOTAL'}},
 'A7': {'titre': '7/ Tableau des immobilisations cedees', 'dynamic': True, 'str_cols': ['date_acquisition'], 'cols': ['date_acquisition', 'montant_net_actif', 'amortissements_pratiques', 'valeur_nette_comptable', 'prix_cession', 'plus_value', 'moins_value'], 'postes': {}},
 'A8': {'titre': '8/ Tableau des provisions et pertes de valeurs', 'cols': ['provisions_cumulees_debut', 'dotations_exercice', 'reprises_exercice', 'provisions_cumulees_fin'], 'postes': {
   'pertes_valeur_stocks': 'Pertes de valeurs sur stocks',
   'pertes_valeur_creances': 'Pertes de valeurs sur creances',
   'pertes_valeur_actions': 'Pertes de valeurs sur actions et parts sociales',
   'provisions_pensions': 'Provisions pour pensions et obligations similaires',
   'provisions_litiges': 'Provisions sur litiges',
   'autres_provisions_personnel': 'Autres provisions liees au personnel',
   'provisions_impots': 'Provisions pour impots',
   'autres_provisions': 'Autres provisions',
   'total': 'TOTAL'}},
 'A81': {'titre': '8/1 Releve des pertes de valeurs sur creances', 'dynamic': True, 'cols': ['valeur_creance', 'perte_valeur_constituee'], 'postes': {}},
 'A82': {'titre': '8/2 Releve des pertes de valeurs sur actions', 'dynamic': True, 'cols': ['valeur_nominale_debut', 'perte_valeur_constituee', 'valeur_nette_comptable'], 'postes': {}},
 'A9': {'titre': '9/ Tableau de determination du resultat fiscal', 'cols': ['montant'], 'postes': {
   'resultat_net_benefice': 'I. Resultat net de l exercice Benefice',
   'resultat_net_perte': 'I. Resultat net de l exercice Perte',
   'charges_immeubles_non_affectes': 'Charges des immeubles non affectes directement a l exploitation',
   'quote_part_cadeaux_publicitaires': 'Quote-part des cadeaux publicitaires non deductibles',
   'quote_part_sponsoring': 'Quote-part du sponsoring et parrainage non deductibles',
   'frais_reception': 'Frais de reception non deductibles',
   'cotisations_dons': 'Cotisations et dons non deductibles',
   'impots_taxes_non_deductibles': 'Impots et taxes non deductibles',
   'provisions_non_deductibles': 'Provisions non deductibles',
   'amortissements_non_deductibles': 'Amortissements non deductibles',
   'quote_part_frais_rd': 'Quote-part des frais de recherche developpement non deductibles',
   'amortissements_credit_bail_preneur': 'Amortissements non deductibles credit bail (Preneur)',
   'loyers_hors_produits_financiers_bailleur': 'Loyers hors produits financiers (bailleur)',
   'ibs_impot_exigible': 'Impots sur les benefices - Impot exigible sur le resultat',
   'ibs_impot_differe': 'Impots sur les benefices - Impot differe (variation)',
   'pertes_valeur_non_deductibles': 'Pertes de valeurs non deductibles',
   'amendes_penalites': 'Amendes et penalites',
   'autres_reintegrations': 'Autres reintegrations',
   'total_reintegrations': 'Total des reintegrations',
   'plus_values_cession_actif_immobilise': 'Plus values sur cession d elements d actif immobilises',
   'produits_plus_values_actions_bourse': 'Produits et plus values de cession des actions et titres assimiles cotes en bourse',
   'revenus_distribution_benefices': 'Revenus provenant de la distribution des benefices',
   'amortissements_credit_bail_bailleur': 'Amortissements credit bail (Bailleur)',
   'loyers_hors_charges_financieres_preneur': 'Loyers hors charges financieres (Preneur)',
   'complement_amortissements': 'Complement d amortissements',
   'autres_deductions': 'Autres deductions',
   'total_deductions': 'Total des deductions',
   'total_deficits_a_deduire': 'Total des deficits a deduire',
   'resultat_fiscal_benefice': 'Resultat fiscal (I+II-III-IV) Benefice',
   'resultat_fiscal_deficit': 'Resultat fiscal (I+II-III-IV) Deficit'}},
 'A10': {'titre': '10/ Tableau d affectation du resultat et des reserves (N-1)', 'cols': ['montant'], 'postes': {
   'origine_report_a_nouveau_n1': 'Report a nouveau de l exercice N-1',
   'origine_resultat_n1': 'Resultat de l exercice N-1',
   'origine_prelevements_reserves': 'Prelevements sur reserves',
   'origine_total': 'TOTAL (origine)',
   'affectation_reserves': 'Reserves',
   'affectation_augmentation_capital': 'Augmentation du capital',
   'affectation_dividendes': 'Dividendes',
   'affectation_report_a_nouveau': 'Report a nouveau',
   'affectation_total': 'TOTAL (affectation)'}},
 'A11': {'titre': '11/ Tableau des participations', 'dynamic': True, 'cols': ['capitaux_propres', 'dont_capital', 'quote_part_capital_pct', 'resultat_dernier_exercice', 'prets_avances', 'dividendes_encaisses', 'valeur_comptable_titres'], 'postes': {}},
 'A12': {'titre': '12/ Commissions courtages redevances honoraires sous-traitance', 'dynamic': True, 'str_cols': ['nif', 'adresse'], 'cols': ['nif', 'adresse', 'montant_percu'], 'postes': {}},
 'A13': {'titre': '13/ Taxe sur l activite professionnelle', 'dynamic': True, 'cols': ['ca_imposable', 'ca_exonere', 'tap_acquittee'], 'postes': {}},
 'ST': {'titre': 'Operations de sous-traitance', 'dynamic': True, 'str_cols': ['nif', 'article', 'adresse', 'reference_contrat'], 'cols': ['nif', 'article', 'adresse', 'reference_contrat', 'montant'], 'postes': {}},
 'REM': {'titre': 'Remunerations versees aux membres de certaines societes', 'dynamic': True, 'str_cols': ['annee_versement'], 'cols': ['nombre_parts', 'annee_versement', 'traitement_emoluments', 'representation_mission_forfait', 'representation_mission_remboursement', 'frais_pro_forfait', 'frais_pro_remboursement'], 'postes': {}},
 'DIST': {'titre': 'Repartition des produits des actions et parts sociales distribues', 'cols': ['montant'], 'postes': {
   'montant_global_brut': 'Montant global brut des distributions',
   'paye_par_societe': 'Paye par la societe elle meme',
   'paye_par_etablissement': 'Paye par un etablissement charge du service des titres',
   'total_revenus_repartis': 'Montant total des revenus repartis'}}
}

ANNEXE_NUM_MAP = {1:'A1', 2:'A2', 3:'A3', 4:'A4', 5:'A5', 6:'A6', 7:'A7', 8:'A8', 9:'A9', 10:'A10', 11:'A11', 12:'A12', 13:'A13'}

# ── Classification 100% contenu : titres imprimes, jamais l'ordre ──
def types_from_title(titre):
    t = ' '.join(strip_accents(titre or '').upper().split())
    if not t:
        return [], []
    if 'BILAN' in t and 'ACTIF' in t:
        return ['ACTIF'], []
    if 'BILAN' in t and 'PASSIF' in t:
        return ['PASSIF'], []
    if 'COMPTE DE RESULTAT' in t or 'REESULTAT' in t:
        return ['TCR'], []
    if 'DECLARATION' in t and ('BENEFICES' in t or 'SOUSCRIPTION' in t or 'TAXE LOCALE' in t):
        return ['DECL'], []
    types, nums = [], []
    for m in re.finditer('(^| )(1[0-3]|[0-9])/([0-9])?(?![0-9])', t):
        n = int(m.group(2))
        s = m.group(3)
        if n == 8 and s in ('1', '2'):
            code = 'A8' + s
        else:
            code = ANNEXE_NUM_MAP.get(n)
        if code and code not in types:
            types.append(code)
            nums.append(n)
    if 'SOUS-TRAITANCE' in t and 'ST' not in types:
        types.append('ST')
    if 'REMUNERATION' in t and ('MEMBRES' in t or 'VERSEE' in t) and 'REM' not in types:
        types.append('REM')
    if 'REPARTITION DES PRODUITS' in t and 'DIST' not in types:
        types.append('DIST')
    return types, nums

PROMPT_CLASSIF = ('Lis cette page scannee d une liasse fiscale algerienne Serie G. ')
PROMPT_CLASSIF += ('Reponds en JSON strict avec exactement deux champs: titre et type. ')
PROMPT_CLASSIF += ('titre = toutes les lignes de titre imprimees de tableaux presentes sur la page, separees par | , en gardant les numeros du type 10/ ou 8/1 tels quels. ')
PROMPT_CLASSIF += ('type = ton hypothese parmi ACTIF, PASSIF, TCR, DECL, ANNEXE, AUTRE.')

RULES = [
 'REGLES: JSON valide uniquement, sans markdown, sans backticks.',
 'Aucune valeur inventee. Case vide ou absent: null. Illisible: null.',
 'Montants en nombres JSON sans separateurs de milliers.',
 'Montant entre parentheses = negatif: (1 553 799) devient -1553799.',
 'Textes (nif, adresses, noms, dates) en chaines telles quelles.'
]

def build_prompt(types):
    L = []
    L.append('Lis cette page scannee d une liasse fiscale algerienne (imprime Serie G).')
    L.append('Extrais en JSON strict les tableaux suivants, identifies par leur code.')
    L.append('Structure: { type_page, numero_page_imprimee, titre_page, entete: {entreprise, nif, exercice, exercice_du, exercice_au, adresse, activite, serie_g}, puis une cle par code de tableau present.')
    for t in types:
        spec = SCHEMAS[t]
        L.append('--- Code ' + t + ' : ' + spec['titre'] + ' ---')
        if spec.get('dynamic'):
            L.append('Tableau a lignes libres. Retourne: {lignes: [ {libelle_imprime, valeurs: {colonne: nombre ou texte ou null}} ]}')
            L.append('Colonnes: ' + ', '.join(spec['cols']))
            L.append('Seulement les lignes non vides; garde les totaux si presents.')
        else:
            L.append('Retourne: {lignes: [ {row_code, libelle_imprime, valeurs: {colonne: nombre ou null}} ]}')
            L.append('Colonnes: ' + ', '.join(spec['cols']))
            if t == 'DECL':
                L.append('Les valeurs peuvent etre des nombres ou des textes selon le poste.')
            L.append('Row_code autorises:')
            for k, lab in spec['postes'].items():
                L.append('- ' + k + ' : ' + lab)
            L.append('Retourne seulement les row_code avec au moins une valeur non nulle.')
    L.extend(RULES)
    return chr(10).join(L)

VALID_CODES = set(SCHEMAS.keys()) | {'AUTRE', 'BLANCHE'}
print('✅ Schemas + classification contenu OK (' + str(len(SCHEMAS)) + ' schemas)')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 8 — NORMALISATION | BILANS_V10
# Objectif: sorties VLM → valeurs typées stables (float/str/null).
# Entrees: bloc JSON par code de tableau.
# Sorties: {key:{col:float}} pour schemas fixes; liste de lignes pour schemas dynamiques.
# Regles: aucune valeur inventee; null si absent.
# ════════════════════════════════════════════════════════════
def norm_str(v):
    if v is None:
        return None
    if isinstance(v, bool):
        return None
    s = ' '.join(str(v).split())
    if not s or s.lower() in ('null', 'none', 'n/a', 'na', '-'):
        return None
    return s

def norm_montant(v):
    if v is None:
        return None
    if isinstance(v, bool):
        return None
    if isinstance(v, (int, float)):
        return float(v)
    s = str(v).strip()
    if s.lower() in ('null', 'none', 'n/a', 'na', '-'):
        return None
    neg = (s.startswith('(') and s.endswith(')')) or s.startswith('-')
    s = re.sub('[^0-9.,-]', '', s)
    if not s:
        return None
    if s.count(',') == 1 and '.' not in s:
        s = s.replace(',', '.')
    elif ',' in s:
        s = s.replace(',', '')
    elif s.count('.') > 1:
        s = s.replace('.', '')
    try:
        return -float(s) if neg else float(s)
    except Exception:
        return None

def norm_nif(v):
    s = norm_str(v)
    return re.sub('[^0-9]', '', s) if s else None

def norm_annee4(v):
    s = norm_str(v) or ''
    m = re.findall('20[0-9]{2}', s)
    return m[-1] if m else None

def norm_by_kind(v, kind):
    if kind == 'texte':
        return norm_str(v)
    if kind == 'nif':
        return norm_nif(v)
    if kind == 'annee':
        return norm_annee4(v)
    return norm_montant(v)

def normalise_fixed(t, block):
    spec = SCHEMAS[t]
    kinds = spec.get('kinds', {})
    lignes = (block or {}).get('lignes') or []
    label_to_key = {norm_key(v): k for k, v in spec['postes'].items()}
    row_map = {}
    for lg in lignes:
        if not isinstance(lg, dict):
            continue
        rc = norm_str(lg.get('row_code'))
        if rc not in spec['postes']:
            lab = norm_str(lg.get('libelle_imprime'))
            if lab and norm_key(lab) in label_to_key:
                rc = label_to_key[norm_key(lab)]
        if rc in spec['postes']:
            row_map[rc] = lg
    out = {}
    for key in spec['postes']:
        vals = (row_map.get(key) or {}).get('valeurs') or {}
        if not isinstance(vals, dict):
            vals = {}
        out[key] = {}
        for col in spec['cols']:
            kind = kinds.get(key, 'montant')
            out[key][col] = norm_by_kind(vals.get(col), kind)
    return out

def normalise_dynamic(t, block):
    spec = SCHEMAS[t]
    str_cols = set(spec.get('str_cols', []))
    out = []
    for lg in (block or {}).get('lignes') or []:
        if not isinstance(lg, dict):
            continue
        vals = lg.get('valeurs') or {}
        if not isinstance(vals, dict):
            vals = {}
        row = {'libelle_imprime': norm_str(lg.get('libelle_imprime'))}
        for col in spec['cols']:
            row[col] = norm_str(vals.get(col)) if col in str_cols else norm_montant(vals.get(col))
        if row['libelle_imprime'] or any(v is not None for k, v in row.items() if k != 'libelle_imprime'):
            out.append(row)
    return out

def normalise_table(t, block):
    if t == 'AUTRE':
        return block or {}
    if SCHEMAS[t].get('dynamic'):
        return normalise_dynamic(t, block)
    return normalise_fixed(t, block)

def normalise_entete(data):
    ent = data.get('entete') or {}
    if not isinstance(ent, dict):
        ent = {}
    return {
        'entreprise': norm_str(ent.get('entreprise')),
        'nif': norm_nif(ent.get('nif')),
        'exercice': norm_str(ent.get('exercice')),
        'exercice_du': norm_str(ent.get('exercice_du')),
        'exercice_au': norm_str(ent.get('exercice_au')),
        'adresse': norm_str(ent.get('adresse')),
        'activite': norm_str(ent.get('activite')),
        'serie_g': norm_str(ent.get('serie_g'))
    }

def count_values(obj):
    count = 0
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k == 'brut':
                continue
            count += count_values(v)
    elif isinstance(obj, list):
        for v in obj:
            count += count_values(v)
    elif isinstance(obj, (int, float)) and not isinstance(obj, bool):
        count += 1
    elif isinstance(obj, str) and obj:
        count += 1
    return count

print('✅ Normalisation OK')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 9 — CONSTRUCTION PAGES JSON | BILANS_V10
# Objectif: construire chaque page du JSON final + synthese.
# Entrees: page PDF, types detectes, donnees VLM.
# Sorties: objet page, synthese, document JSON.
# Regles: page scannee toujours tracee; pages blanches sans VLM.
# ════════════════════════════════════════════════════════════
def build_blank_page(page, fichier_source):
    page_number = page['index'] + 1
    return {
        'page_id': 'p' + str(page_number).zfill(3),
        'pdf_page_index': page['index'],
        'numero_page_scannee': page_number,
        'numero_page_imprimee': None,
        'fichier_source': fichier_source,
        'classification': {'type_page': 'BLANCHE', 'types': [], 'annexe_numeros': [], 'sous_type_page': 'PAGE_VIERGE', 'titre_page': None, 'page_annexe': False, 'page_utile': False, 'page_blanche': True},
        'image': {'largeur_px': page.get('largeur_px'), 'hauteur_px': page.get('hauteur_px'), 'rotation_estimee': page.get('rotation_estimee'), 'deskew_applique': bool(page.get('rotation_estimee'))},
        'statut_extraction': {'statut': 'BLANCHE', 'nb_tableaux': 0, 'nb_champs_extraits': 0, 'commentaire': 'Page blanche detectee, non envoyee au VLM.'},
        'entete_page': {},
        'donnees': {},
        'tokens_in': 0, 'tokens_out': 0, 'temps_s': 0.0
    }

def build_page_object(page, types, data, rep, fichier_source):
    page_number = page['index'] + 1
    data = data if isinstance(data, dict) else {}
    donnees = {'brut': data}
    for t in types:
        if t == 'AUTRE':
            donnees['AUTRE'] = {'tables': data.get('tables') or []}
        else:
            donnees[t] = normalise_table(t, data.get(t))
    statut = 'OK' if data else 'ECHEC_EXTRACTION'
    nb_champs = count_values(donnees)
    nums = page.get('annexe_nums') or []
    sous_type = '_'.join(types) if types else 'AUTRE'
    return {
        'page_id': 'p' + str(page_number).zfill(3),
        'pdf_page_index': page['index'],
        'numero_page_scannee': page_number,
        'numero_page_imprimee': data.get('numero_page_imprimee') if isinstance(data.get('numero_page_imprimee'), int) else None,
        'fichier_source': fichier_source,
        'classification': {
            'type_page': types[0] if len(types) == 1 else (types[0] if types else 'AUTRE'),
            'types': types,
            'annexe_numeros': nums,
            'sous_type_page': sous_type,
            'titre_page': norm_str(data.get('titre_page')),
            'page_annexe': any(t.startswith('A') or t in ('ST', 'REM', 'DIST') for t in types),
            'page_utile': bool(types),
            'page_blanche': False
        },
        'image': {'largeur_px': page.get('largeur_px'), 'hauteur_px': page.get('hauteur_px'), 'rotation_estimee': page.get('rotation_estimee'), 'deskew_applique': bool(page.get('rotation_estimee'))},
        'statut_extraction': {'statut': statut, 'nb_tableaux': len(types), 'nb_champs_extraits': nb_champs, 'commentaire': None},
        'entete_page': normalise_entete(data),
        'donnees': donnees,
        'tokens_in': rep.get('tokens_in') if rep else 0,
        'tokens_out': rep.get('tokens_out') if rep else 0,
        'temps_s': rep.get('elapsed') if rep else 0.0
    }

def merge_core(base, new):
    if new is None:
        return base
    if base is None:
        return copy.deepcopy(new)
    for key, cols in new.items():
        if key not in base:
            base[key] = copy.deepcopy(cols)
        elif isinstance(cols, dict):
            for col, val in cols.items():
                if base[key].get(col) is None and val is not None:
                    base[key][col] = val
    return base

def build_synthese(page_objects):
    synthese = {'actif': None, 'passif': None, 'tcr': None, 'decl': None, 'annexes': {}, 'annexes_pages': []}
    for page in page_objects:
        types = page['classification'].get('types') or []
        donnees = page.get('donnees', {})
        if 'ACTIF' in types and synthese['actif'] is None:
            synthese['actif'] = donnees.get('ACTIF')
        if 'PASSIF' in types and synthese['passif'] is None:
            synthese['passif'] = donnees.get('PASSIF')
        if 'TCR' in types:
            synthese['tcr'] = merge_core(synthese['tcr'], donnees.get('TCR'))
        if 'DECL' in types and synthese['decl'] is None:
            synthese['decl'] = donnees.get('DECL')
        for t in types:
            if t in ('ACTIF', 'PASSIF', 'TCR', 'DECL', 'AUTRE'):
                continue
            if synthese['annexes'].get(t) is None:
                synthese['annexes'][t] = donnees.get(t)
        if any(t not in ('ACTIF', 'PASSIF', 'TCR', 'DECL', 'AUTRE') for t in types):
            synthese['annexes_pages'].append({'page_id': page['page_id'], 'numero_page_scannee': page['numero_page_scannee'], 'types': types, 'annexe_numeros': page['classification'].get('annexe_numeros')})
    return synthese

def build_document(pdf_path, pages, page_objects, tok_in, tok_out, elapsed):
    nb_pages = len(pages)
    nb_blanches = sum(1 for p in page_objects if p['classification']['type_page'] == 'BLANCHE')
    nb_utiles = sum(1 for p in page_objects if p['classification']['type_page'] != 'BLANCHE')
    nb_annexes = sum(1 for p in page_objects if p['classification'].get('page_annexe'))
    return {
        'schema_version': '10.0',
        'type_document': 'liasse_fiscale_algerienne_serie_g',
        'document': {
            'document_id': 'doc_' + datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + pdf_path.stem,
            'fichier_source': pdf_path.name,
            'nb_pages_pdf': nb_pages,
            'nb_pages_scannes': nb_pages,
            'nb_pages_blanches': nb_blanches,
            'nb_pages_utiles': nb_utiles,
            'nb_pages_annexes': nb_annexes,
            'date_extraction': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'duree_extraction_s': round(elapsed, 2),
            'modele_extraction': MODEL_PATH.split('/')[-1],
            'prompt_version': 'v10.annexes_structurees',
            'referentiel_version': 'liasse_serie_g_v10',
            'tokens_in': tok_in,
            'tokens_out': tok_out,
            'tokens_total': tok_in + tok_out
        },
        'pages': page_objects,
        'synthese': build_synthese(page_objects)
    }

print('✅ Construction pages JSON OK')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 10 — PIPELINE PRINCIPAL | BILANS_V10
# Objectif: executer le pipeline PDF vers JSON avec progression live.
# Entrees: PDF dans INPUT_DIR.
# Sorties: JSON par dossier + index JSONL + log detaille.
# Regles: checkpoint JSON; pages blanches sans VLM; arbitre contenu;
#         traceback complet dans le log en cas d erreur.
# ════════════════════════════════════════════════════════════
deja = {f.stem for f in JSON_DIR.glob('*.json')}
a_traiter = [p for p in pdfs if p.stem not in deja]
log('A traiter: ' + str(len(a_traiter)) + ' | deja traites: ' + str(len(deja)))

t_total = time.time()
n_ok = 0
n_err = 0
index_records = []
prompt_cache = {}

for num, pdf_path in enumerate(a_traiter, start=1):
    prefix = '[' + str(num).zfill(4) + '/' + str(len(a_traiter)) + '] '
    t_pdf = time.time()
    log(prefix + 'DEMARRAGE ' + pdf_path.name)
    try:
        t0 = time.time()
        pages = pdf_to_pages(pdf_path)
        log(prefix + 'rendu termine: ' + str(len(pages)) + ' pages en ' + str(round(time.time()-t0, 1)) + 's')

        page_objects = []
        active_pages = []
        for p in pages:
            if is_blank(p['image']):
                page_objects.append(build_blank_page(p, pdf_path.name))
            else:
                active_pages.append(p)
        log(prefix + 'pages blanches: ' + str(len(page_objects)) + ' | pages actives: ' + str(len(active_pages)))

        tok_in = 0
        tok_out = 0

        if active_pages:
            # ── PASS 1 : classification par contenu (titres) ──
            t0 = time.time()
            mini = [resize(p['image'], 600) for p in active_pages]
            reps1 = []
            for bs in chunks(mini, CLASSIF_BATCH_SIZE):
                reps1 += ask_batch(PROMPT_CLASSIF, bs)
            tok_in += sum(r['tokens_in'] for r in reps1)
            tok_out += sum(r['tokens_out'] for r in reps1)
            log(prefix + 'classification terminee en ' + str(round(time.time()-t0, 1)) + 's')

            pages_typed = []
            type_counts = {}
            for p, rep in zip(active_pages, reps1):
                d = parse_json(rep['text'])
                titre = norm_str(d.get('titre'))
                t_model = (norm_str(d.get('type')) or '').upper()
                types, nums = types_from_title(titre)
                if not types:
                    if t_model in ('ACTIF', 'PASSIF', 'TCR', 'DECL'):
                        types = [t_model]
                    elif t_model == 'ANNEXE':
                        types = ['AUTRE']
                    else:
                        types = ['AUTRE']
                p['types'] = types
                p['annexe_nums'] = nums
                pages_typed.append((p, types))
                label = '+'.join(types)
                type_counts[label] = type_counts.get(label, 0) + 1
            log(prefix + 'types (contenu): ' + ', '.join(str(k) + '=' + str(v) for k, v in sorted(type_counts.items())))

            del mini, reps1
            gc.collect()

            # ── PASS 2 : extraction structuree par groupe de types ──
            groups = {}
            for p, types in pages_typed:
                groups.setdefault(tuple(types), []).append(p)

            extracted = {}
            for key_types, plist in groups.items():
                prompt = prompt_cache.get(key_types)
                if prompt is None:
                    prompt = build_prompt(list(key_types)) if key_types != ('AUTRE',) else build_prompt(['A13'])
                    if key_types == ('AUTRE',):
                        prompt = ('Lis cette page d un dossier fiscal algerien. Extrais en JSON strict: type_page, titre_page, entete, tables (liste de {table_libelle, colonnes, lignes}). ' + chr(10).join(RULES))
                    prompt_cache[key_types] = prompt
                nb_batches = (len(plist) + GPU_BATCH_SIZE - 1) // GPU_BATCH_SIZE
                batch_no = 0
                for batch in chunks(plist, GPU_BATCH_SIZE):
                    batch_no += 1
                    page_nums = [str(p['index'] + 1) for p in batch]
                    log(prefix + 'extraction ' + '+'.join(key_types) + ' batch ' + str(batch_no) + '/' + str(nb_batches) + ' pages [' + ','.join(page_nums) + ']')
                    t0 = time.time()
                    reps = ask_batch(prompt, [p['image'] for p in batch])
                    for p, rep in zip(batch, reps):
                        tok_in += rep['tokens_in']
                        tok_out += rep['tokens_out']
                        data = parse_json(rep['text'])
                        extracted[p['index']] = build_page_object(p, list(key_types), data, rep, pdf_path.name)
                    log(prefix + 'extraction ' + '+'.join(key_types) + ' batch ' + str(batch_no) + ' terminee en ' + str(round(time.time()-t0, 1)) + 's | tokens=' + str(sum(r['tokens_in'] + r['tokens_out'] for r in reps)))
                    gc.collect()
                    torch.cuda.empty_cache()

            for p in active_pages:
                if p['index'] in extracted:
                    page_objects.append(extracted[p['index']])
                else:
                    page_objects.append(build_page_object(p, ['AUTRE'], {}, None, pdf_path.name))

        page_objects.sort(key=lambda x: x['pdf_page_index'])
        elapsed = time.time() - t_pdf

        result = build_document(pdf_path, pages, page_objects, tok_in, tok_out, elapsed)

        json_path = JSON_DIR / (pdf_path.stem + '.json')
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(result, f, ensure_ascii=False, indent=2, default=str)

        n_ok += 1
        index_records.append({
            'fichier': pdf_path.name,
            'json_path': str(json_path),
            'nb_pages': len(pages),
            'nb_pages_utiles': result['document']['nb_pages_utiles'],
            'nb_pages_blanches': result['document']['nb_pages_blanches'],
            'nb_pages_annexes': result['document']['nb_pages_annexes'],
            'tokens_total': tok_in + tok_out,
            'duree_s': round(elapsed, 2),
            'date_extraction': result['document']['date_extraction']
        })

        eta = (time.time() - t_total) / num * (len(a_traiter) - num)
        log(prefix + 'OK ' + pdf_path.name + ' | ' + str(round(elapsed, 1)) + 's | tok=' + str(tok_in + tok_out) + ' | pages=' + str(len(pages)) + ' | utiles=' + str(result['document']['nb_pages_utiles']) + ' | blanches=' + str(result['document']['nb_pages_blanches']) + ' | annexes=' + str(result['document']['nb_pages_annexes']) + ' | ETA ' + str(round(eta/3600, 2)) + 'h')

    except Exception as e:
        n_err += 1
        import traceback as _tb
        log(prefix + 'ERREUR ' + pdf_path.name + ' — ' + type(e).__name__ + ': ' + str(e))
        with open(LOG_PATH, 'a', encoding='utf-8') as f:
            f.write(_tb.format_exc() + chr(10))
        continue

with open(INDEX_PATH, 'w', encoding='utf-8') as f:
    for rec in index_records:
        f.write(json.dumps(rec, ensure_ascii=False) + chr(10))

log('✅ Termine en ' + str(round(time.time() - t_total, 1)) + 's | OK ' + str(n_ok) + ' | Erreurs ' + str(n_err))
log('Index JSON: ' + str(INDEX_PATH))

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 11 — VALIDATION RAPIDE | BILANS_V10
# Objectif: relire le dernier JSON et verifier la structure page par page.
# Entrees: JSON_DIR.
# Sorties: resume pages/types/annexes/champs.
# Regles: lecture seule, aucune modification.
# ════════════════════════════════════════════════════════════
files = sorted(JSON_DIR.glob('*.json'))
if not files:
    print('Aucun JSON pour le moment.')
else:
    d = json.load(open(files[-1], encoding='utf-8'))
    print('Fichier:', d['document']['fichier_source'])
    print('Pages:', d['document']['nb_pages_pdf'], '| utiles:', d['document']['nb_pages_utiles'], '| blanches:', d['document']['nb_pages_blanches'], '| annexes:', d['document']['nb_pages_annexes'])
    print('Tokens:', d['document']['tokens_total'], '| Duree:', d['document']['duree_extraction_s'], 's')
    for p in d['pages']:
        c = p['classification']
        print(p['page_id'], '| scan', p['numero_page_scannee'], '|', '+'.join(c.get('types') or [c['type_page']]), '| nums', c.get('annexe_numeros'), '| champs:', p['statut_extraction']['nb_champs_extraits'])
    syn = d['synthese']
    print('Synthese: actif', syn['actif'] is not None, '| passif', syn['passif'] is not None, '| tcr', syn['tcr'] is not None, '| decl', syn['decl'] is not None)
    print('Annexes presentes:', ', '.join(sorted(k for k, v in syn['annexes'].items() if v is not None)) or 'aucune')